# 04 · Entity Behavior Analysis

**Amaç:** AccountNumber, DeviceId, ReceiverName, IP_Subnet davranışlarını anlayarak feature engineering & memorization riskini şekillendirmek.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated


In [ ]:
df, _ = load_validated()

def entity_summary(col):
    g = df.groupby(col).agg(n=('IsFraudTransaction','size'), fraud=('IsFraudTransaction','sum'))
    return {
        'unique': len(g),
        'tx_p50': float(g['n'].median()),
        'tx_p90': float(g['n'].quantile(0.9)),
        'tx_p99': float(g['n'].quantile(0.99)),
        'tx_max': int(g['n'].max()),
        'with_fraud': int((g['fraud']>0).sum()),
        'only_fraud': int(((g['fraud']==g['n']) & (g['n']>=1)).sum()),
    }

summary = pd.DataFrame({col: entity_summary(col) for col in ['AccountNumber','DeviceId','ReceiverName','IP_Subnet','SenderName','CustomerName']}).T
summary

## AccountNumber → tx/account dağılımı (ezici çoğunluk 1 tx)

In [ ]:
acc_n = df.groupby('AccountNumber').size()
fig, ax = plt.subplots(figsize=(10,4))
acc_n.clip(upper=10).value_counts().sort_index().plot.bar(ax=ax)
ax.set_title('tx/account (clipped at 10)')
ax.set_xlabel('# transactions per account')
ax.set_ylabel('# accounts')

## DeviceId → distinct accounts per device

In [ ]:
dist = df.groupby('DeviceId')['AccountNumber'].nunique()
print('Cihazların %88'i ≥3 hesap, %66'sı ≥10 hesap kullanıyor:')
print('≥3 acc cihaz:', int((dist>=3).sum()), '/', len(dist))
print('≥10 acc cihaz:', int((dist>=10).sum()), '/', len(dist))
fig, ax = plt.subplots(figsize=(10,4))
dist.clip(upper=50).hist(bins=50, ax=ax)
ax.set_title('Distinct accounts per DeviceId (clip 50)')

## ReceiverName → fraud-only receiver pattern

In [ ]:
rec = df.groupby('ReceiverName').agg(n=('IsFraudTransaction','size'), fraud=('IsFraudTransaction','sum'))
rec['rate'] = rec['fraud']/rec['n']
print('Tamamen fraud receiver sayısı:', int((rec['rate']==1).sum()))
print('≥10 tx alan receiver içinde fraud rate ≥%50:', int(((rec['n']>=10)&(rec['rate']>=0.5)).sum()))

## Sonuç
- AccountNumber'da geçmiş yok → hesap-bazlı historical FE ÜRETİLEMEZ.
- DeviceId güçlü potansiyel — ama ham ID asla modele girmemeli (memorization).
- ReceiverName: 576 'tamamen fraud' receiver. Time-cutoff'lu historical fraud rate güçlü ama label delay riski taşır.